In [63]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from imblearn.over_sampling import SMOTE
import nltk
import re
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.pipeline import Pipeline

In [ ]:
# data preprocessing

In [74]:
df=pd.read_csv('spam.csv',encoding='ISO-8859-1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [ ]:
here columns Unnamed: 2	Unnamed: 3	Unnamed: 4 not required .so we drop it.

In [75]:
df.drop(columns=['Unnamed: 2'	,'Unnamed: 3'	,'Unnamed: 4'],axis=1,inplace=True)

In [8]:
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [76]:
df.v1=df.v1.map({'ham':0,'spam':1})

In [77]:
words=nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [78]:
stop=set(stopwords.words('english'))

In [30]:
# preprocessing the text
cleaned_text=[]
def preprocessing(text):
    for word in text:
        lower=word.lower()
        clean=re.sub(r'[^a-z\s]','',lower)
        split=clean.split()
        cleaned=[i for i in split if i not in stop]
        join=''.join(cleaned)
        cleaned_text.append(join)
    return cleaned_text   

In [79]:
cleaned_text=[]
for word in df.v2:
    lower=word.lower()
    clean=re.sub(r'[^a-z\s]','',lower)
    split=clean.split()
    cleaned= [i for i in split if i not in stop]
    join=' '.join(cleaned)
    cleaned_text.append(join)

In [40]:
cleaned_text

['go jurong point crazy available bugis n great world la e buffet cine got amore wat',
 'ok lar joking wif u oni',
 'free entry wkly comp win fa cup final tkts st may text fa receive entry questionstd txt ratetcs apply overs',
 'u dun say early hor u c already say',
 'nah dont think goes usf lives around though',
 'freemsg hey darling weeks word back id like fun still tb ok xxx std chgs send rcv',
 'even brother like speak treat like aids patent',
 'per request melle melle oru minnaminunginte nurungu vettam set callertune callers press copy friends callertune',
 'winner valued network customer selected receivea prize reward claim call claim code kl valid hours',
 'mobile months u r entitled update latest colour mobiles camera free call mobile update co free',
 'im gonna home soon dont want talk stuff anymore tonight k ive cried enough today',
 'six chances win cash pounds txt csh send cost pday days tsandcs apply reply hl info',
 'urgent week free membership prize jackpot txt word clai

In [80]:
df['cleaned_text']=cleaned_text

In [81]:
df.rename(columns={'v1':'labels'},inplace=True)

In [82]:
df

,labels,v2,cleaned_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,0,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah dont think goes usf lives around though
...,...,...,...
5567,1,This is the 2nd time we have tried 2 contact u...,nd time tried contact u u pound prize claim ea...
5568,0,Will Ì_ b going to esplanade fr home?,b going esplanade fr home
5569,0,"Pity, * was in mood for that. So...any other s...",pity mood soany suggestions
5570,0,The guy did some bitching but I acted like i'd...,guy bitching acted like id interested buying s...


In [83]:
df.drop('v2',axis=1,inplace=True)

In [84]:
df

,labels,cleaned_text
0,0,go jurong point crazy available bugis n great ...
1,0,ok lar joking wif u oni
2,1,free entry wkly comp win fa cup final tkts st ...
3,0,u dun say early hor u c already say
4,0,nah dont think goes usf lives around though
...,...,...
5567,1,nd time tried contact u u pound prize claim ea...
5568,0,b going esplanade fr home
5569,0,pity mood soany suggestions
5570,0,guy bitching acted like id interested buying s...


In [85]:
# convert text into numerical by using countvectorizer or tfidfvectorizer

tfidf=TfidfVectorizer()
x=tfidf.fit_transform(df['cleaned_text'])
y=df['labels']

In [121]:
# splitting data for training ,testing
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,stratify=y,random_state=32)

In [87]:
# building the model
multinb=MultinomialNB()

In [88]:
# train the model
multinb.fit(x_train,y_train)

MultinomialNB()

In [89]:
pred=multinb.predict(x_test)

In [90]:
print(classification_report(y_test,pred))

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       966
           1       1.00      0.76      0.86       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115



In [91]:
confusion_matrix(y_test,pred)

array([[966,   0],
       [ 36, 113]], dtype=int64)

In [92]:
# build the pipeline
pipe=Pipeline([('tfidfvectorizer',TfidfVectorizer()),('model',MultinomialNB())])

In [93]:
# x=df['cleaned_text']
# y=df['labels']

In [122]:
pipe.fit(x_train,y_train)

Pipeline(steps=[('tfidfvectorizer', TfidfVectorizer()),
                ('model', MultinomialNB())])

In [98]:
pipe.predict(['congratulation you won 1 lakh prize '])


array([1], dtype=int64)

In [100]:
# top 10 spam words
feature_names=tfidf.get_feature_names_out()

In [103]:
clasess=multinb.classes_

In [108]:
spam_index=list(clasess).index(1)

In [115]:
spam_proba=multinb.feature_log_prob_[spam_index]

In [118]:
top=10
top_10_spam_index=np.argsort(spam_proba)[-top:]

In [119]:
feature_names[top_10_spam_index]

array(['reply', 'stop', 'text', 'prize', 'ur', 'mobile', 'claim', 'txt',
       'free', 'call'], dtype=object)

In [124]:
pred=pipe.predict(x_test)

In [125]:
print(classification_report(y_test,pred))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98       966
           1       0.99      0.79      0.88       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115



In [126]:
confusion_matrix(y_test,pred)

array([[965,   1],
       [ 32, 117]], dtype=int64)